# 22 - Online per-chunk selector worker 0

This is the causal online aggregation experiment. At every 10-action decision chunk, both arms
generate two clean candidates from the same live observation and execute the lower-uncertainty
candidate:

- `source, 2 queries`: two independent source-checkpoint queries;
- `source + model 1`: one source query and one v2 model-1 query.

Both use K=5 uncertainty at Euler steps `(3,4)` and matched candidate-slot seeds. The two arms
start from exactly the same LIBERO-PRO identity, although their states may diverge after executing
different chunks. Candidate uncertainties, choices, clean-action disagreement metrics, and raw
candidate chunks are logged. No Supabase migration is required.

This notebook is shard 0 of 4. It is resumable and uses 10 episodes per task, so all
four workers together produce 1,300 matched identities per arm. Before committing compute, worker
0 can be run once with `EPISODE_LIMIT=1`; after those two rollouts succeed, set it back to `None`
and rerun. Those smoke-test rows are reused rather than repeated. Use an A100 if available because
source+m1 keeps two pi0.5 policies resident simultaneously; an L4 is not assumed to fit.

## 1. Setup a fresh GPU runtime

In [ ]:
EXTRAS = 'sim'
SETUP_ENV = True
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

## 2. Configuration and resumable collection

In [ ]:
from pathlib import Path
from google.colab import drive
from pnp.config import PI05_REPO_ID
from pnp.diversity import (DIVERSITY_CHUNK_SELECTOR_EXPERIMENT,
    DIVERSITY_V2_EXPERIMENT_PREFIX, load_bootstrap_manifest,
    run_diversity_chunk_selector_worker)

drive.mount("/content/drive")

EPISODES_PER_TASK = 10
SHARD_COUNT = 4
SHARD_INDEX = 0
EPISODE_LIMIT = None  # optional worker-0 smoke test: set 1, run, then restore None
EXPERIMENT = DIVERSITY_CHUNK_SELECTOR_EXPERIMENT
MANIFEST_PATH = Path(
    "/content/drive/MyDrive/pnp_diversity_v2/bootstrap_manifest_finetuned_v2.json")
manifest = load_bootstrap_manifest(MANIFEST_PATH)
assert manifest["source_model"] == PI05_REPO_ID, manifest["source_model"]

print({"experiment": EXPERIMENT, "episodes_per_task": EPISODES_PER_TASK,
       "shard_count": SHARD_COUNT, "shard_index": SHARD_INDEX,
       "episode_limit": EPISODE_LIMIT,
       "manifest_hash": manifest["manifest_hash"]})
run_diversity_chunk_selector_worker(
    episodes_per_task=EPISODES_PER_TASK,
    episode_limit=EPISODE_LIMIT,
    shard_count=SHARD_COUNT, shard_index=SHARD_INDEX,
    manifest_hash=manifest["manifest_hash"],
    diversity_experiment_prefix=DIVERSITY_V2_EXPERIMENT_PREFIX,
    experiment=EXPERIMENT)